# Integrated scGPT Perturbation Pipeline

This notebook combines custom data loading and preprocessing with the scGPT fine-tuning pipeline.

In [1]:
import os
import sys
import time
import copy
import random
import json
import warnings
from pathlib import Path
from typing import Iterable, List, Tuple, Dict, Union, Optional

import numpy as np
import torch
import scanpy as sc
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from torch import nn
from torch.nn import functional as F
from torchtext.vocab import Vocab
from torchtext._torchtext import Vocab as VocabPybind
from torch_geometric.loader import DataLoader

from gears import PertData, GEARS
from gears.inference import compute_metrics, deeper_analysis, non_dropout_analysis
from gears.utils import create_cell_graph_dataset_for_prediction

import scgpt as scg
from scgpt.model import TransformerGenerator
from scgpt.tokenizer import GeneVocab
from scgpt.loss import masked_mse_loss
from scgpt.utils import set_seed, map_raw_id_to_vocab_id, compute_perturbation_metrics

matplotlib.rcParams["savefig.transparent"] = False
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def setup_environment(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    sc.settings.seed = seed
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Random seed set as {seed}")

setup_environment(42)
print(f"\u2705 Research Environment Initialized on {device}")


/conda_envs/scgpt/lib/python3.10/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/conda_envs/scgpt/lib/python3.10/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/conda_envs/scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See htt

Random seed set as 42
✅ Research Environment Initialized on cuda


In [2]:
# --- Settings & Hyperparameters ---
data_name = "norman"
model_dir = "../../model_checkpoints/scGPT_human"
load_model = model_dir

pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
pad_value = 0
pert_pad_id = 0
include_zero_gene = "all"
max_seq_len = 1536
n_bins = 51

epochs = 15
lr = 1e-4
batch_size = 64
eval_batch_size = 64
schedule_interval = 1
early_stop = 10
amp = True

embsize = 128
d_hid = 512
nlayers = 4
nhead = 4
n_layers_cls = 3
dropout = 0.2
use_fast_transformer = False

MLM = True
CLS = False
CCE = False
MVC = False
ECS = False

log_interval = 5
perts_to_plot = ["SAMD1+ZBTB1"] 

save_dir = Path(f"./save/dev_perturb_{data_name}-{time.strftime('%b%d-%H-%M')}/")
save_dir.mkdir(parents=True, exist_ok=True)
print(f"saving to {save_dir}")

logger = scg.logger
scg.utils.add_file_handler(logger, save_dir / "run.log")
logger.info(f"Running on {time.strftime('%Y-%m-%d %H:%M:%S')}")


saving to save/dev_perturb_norman-Mar04-23-30
scGPT - INFO - Running on 2026-03-04 23:30:15


In [3]:
# --- 1. Data Preparation & scGPT Loading ---
adata = sc.read_h5ad("../../data/splits/norman_2019_raw_sparse_with_splits_original_from_GEARS.h5ad")

# Apply exact splitting mappings
adata.obs['split'] = adata.obs['gears_split']

# Apply scanning preprocessing
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=1200, subset=True)

print(adata)

# Pass perfectly preprocessed AnnData into the PertData framework 
pert_data = PertData("./data")
pert_data.new_data_process(dataset_name="norman_custom_split", adata=adata)

# Automatically uses adata.obs['split'] setup above
pert_data.get_dataloader(batch_size=batch_size, test_batch_size=eval_batch_size)


AnnData object with n_obs × n_vars = 89357 × 1200
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name', 'split', 'gears_split'
    var: 'gene_name', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20', 'log1p', 'hvg'
    layers: 'counts'


Creating pyg object for each cell in the data...
  0%|          | 1/277 [00:00<03:50,  1.20it/s]


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# --- 2. Model Weight & Vocab Initialization ---
load_param_prefixs = ["encoder", "value_encoder", "transformer_encoder"]
model_file = Path(load_model) / "best_model.pt"
vocab_file = Path(load_model) / "vocab.json"

vocab = GeneVocab.from_file(vocab_file)
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

pert_data.adata.var["id_in_vocab"] = [
    1 if gene in vocab else -1 for gene in pert_data.adata.var["gene_name"]
]
gene_ids_in_vocab = np.array(pert_data.adata.var["id_in_vocab"])
logger.info(
    f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
    f"in vocabulary of size {len(vocab)}."
)

genes = pert_data.adata.var["gene_name"].tolist()
gene_ids = np.array(
    [vocab[gene] if gene in vocab else vocab["<pad>"] for gene in genes], dtype=int
)
n_genes = len(genes)

ntokens = len(vocab)
model = TransformerGenerator(
    ntokens, embsize, nhead, d_hid, nlayers,
    nlayers_cls=n_layers_cls, n_cls=1, vocab=vocab, dropout=dropout,
    pad_token=pad_token, pad_value=pad_value, pert_pad_id=pert_pad_id,
    use_fast_transformer=use_fast_transformer,
)

pretrained_dict = torch.load(model_file)
model_dict = model.state_dict()
filtered_dict = {
    k: v for k, v in pretrained_dict.items()
    if any(k.startswith(prefix) for prefix in load_param_prefixs)
    and k in model_dict and v.shape == model_dict[k].shape
}

missing_keys, unexpected_keys = model.load_state_dict(filtered_dict, strict=False)
logger.info(f"Loaded {len(filtered_dict)} parameters.")
model.to(device)


In [ ]:
# --- 3. Training and Evaluation Functions ---
criterion = masked_mse_loss
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, schedule_interval, gamma=0.9)
scaler = torch.cuda.amp.GradScaler(enabled=amp)

def train(model: nn.Module, train_loader: DataLoader) -> None:
    model.train()
    total_loss, total_mse = 0.0, 0.0
    start_time = time.time()
    num_batches = len(train_loader)

    for batch, batch_data in enumerate(train_loader):
        batch_size = len(batch_data.y)
        batch_data.to(device)
        x: torch.Tensor = batch_data.x
        ori_gene_values = x[:, 0].view(batch_size, n_genes)
        pert_flags = x[:, 1].long().view(batch_size, n_genes)
        target_gene_values = batch_data.y

        if include_zero_gene == "all":
            input_gene_ids = torch.arange(n_genes, device=device, dtype=torch.long)
        else:
            input_gene_ids = ori_gene_values.nonzero()[:, 1].flatten().unique().sort()[0]

        if len(input_gene_ids) > max_seq_len:
            input_gene_ids = torch.randperm(len(input_gene_ids), device=device)[:max_seq_len]
            
        input_values = ori_gene_values[:, input_gene_ids]
        input_pert_flags = pert_flags[:, input_gene_ids]
        target_values = target_gene_values[:, input_gene_ids]
        mapped_input_gene_ids = map_raw_id_to_vocab_id(input_gene_ids, gene_ids).repeat(batch_size, 1)
        src_key_padding_mask = torch.zeros_like(input_values, dtype=torch.bool, device=device)

        with torch.cuda.amp.autocast(enabled=amp):
            output_dict = model(
                mapped_input_gene_ids, input_values, input_pert_flags,
                src_key_padding_mask=src_key_padding_mask,
                CLS=CLS, CCE=CCE, MVC=MVC, ECS=ECS,
            )
            output_values = output_dict["mlm_output"]
            masked_positions = torch.ones_like(input_values, dtype=torch.bool)
            loss = loss_mse = criterion(output_values, target_values, masked_positions)

        model.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_mse += loss_mse.item()
        if batch % log_interval == 0 and batch > 0:
            cur_lr = scheduler.get_last_lr()[0]
            ms_per_batch = (time.time() - start_time) * 1000 / log_interval
            cur_loss, cur_mse = total_loss / log_interval, total_mse / log_interval
            logger.info(
                f"| epoch {epoch:3d} | {batch:3d}/{num_batches:3d} batches | "
                f"lr {cur_lr:05.4f} | ms/batch {ms_per_batch:5.2f} | "
                f"loss {cur_loss:5.2f} | mse {cur_mse:5.2f} |"
            )
            total_loss, total_mse = 0, 0
            start_time = time.time()

def eval_perturb(loader: DataLoader, model: TransformerGenerator, device: torch.device) -> Dict:
    model.eval()
    model.to(device)
    pert_cat, pred, truth, pred_de, truth_de = [], [], [], [], []
    results = {}

    for batch in loader:
        batch.to(device)
        pert_cat.extend(batch.pert)
        with torch.no_grad():
            p = model.pred_perturb(batch, include_zero_gene=include_zero_gene, gene_ids=gene_ids)
            t = batch.y
            pred.extend(p.cpu())
            truth.extend(t.cpu())
            for itr, de_idx in enumerate(batch.de_idx):
                pred_de.append(p[itr, de_idx])
                truth_de.append(t[itr, de_idx])

    results["pert_cat"] = np.array(pert_cat)
    results["pred"] = torch.stack(pred).detach().cpu().numpy().astype(np.float64)
    results["truth"] = torch.stack(truth).detach().cpu().numpy().astype(np.float64)
    results["pred_de"] = torch.stack(pred_de).detach().cpu().numpy().astype(np.float64)
    results["truth_de"] = torch.stack(truth_de).detach().cpu().numpy().astype(np.float64)
    return results


In [ ]:
# --- 4. Training Loop ---
best_val_corr = 0
best_model = None
patience = 0

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    train_loader = pert_data.dataloader["train_loader"]
    valid_loader = pert_data.dataloader["val_loader"]

    train(model, train_loader)

    val_res = eval_perturb(valid_loader, model, device)
    val_metrics = compute_perturbation_metrics(
        val_res, pert_data.adata[pert_data.adata.obs["condition"] == "ctrl"]
    )
    logger.info(f"val_metrics at epoch {epoch}: \n{val_metrics}")

    elapsed = time.time() - epoch_start_time
    logger.info(f"| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | ")

    val_score = val_metrics["pearson"]
    if val_score > best_val_corr:
        best_val_corr = val_score
        best_model = copy.deepcopy(model)
        logger.info(f"Best model with score {val_score:5.4f}")
        patience = 0
    else:
        patience += 1
        if patience >= early_stop:
            logger.info(f"Early stop at epoch {epoch}")
            break

    scheduler.step()

if best_model is not None:
    torch.save(best_model.state_dict(), save_dir / "best_model.pt")


In [ ]:
# --- 5. Analysis & Plotting ---
def predict(model: TransformerGenerator, pert_list: List[str], pool_size: Optional[int] = None) -> Dict:
    adata = pert_data.adata
    ctrl_adata = adata[adata.obs["condition"] == "ctrl"]
    if pool_size is None:
        pool_size = len(ctrl_adata.obs)
    gene_list = pert_data.gene_names.values.tolist()
    
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        results_pred = {}
        for pert in pert_list:
            cell_graphs = create_cell_graph_dataset_for_prediction(
                pert, ctrl_adata, gene_list, device, num_samples=pool_size
            )
            loader = DataLoader(cell_graphs, batch_size=eval_batch_size, shuffle=False)
            preds = []
            for batch_data in loader:
                pred_gene_values = model.pred_perturb(batch_data, include_zero_gene, gene_ids=gene_ids, amp=amp)
                preds.append(pred_gene_values)
            preds = torch.cat(preds, dim=0)
            results_pred["_".join(pert)] = np.mean(preds.detach().cpu().numpy(), axis=0)
    return results_pred

test_loader = pert_data.dataloader["test_loader"]
test_res = eval_perturb(test_loader, best_model, device)
test_metrics = compute_perturbation_metrics(
    test_res, pert_data.adata[pert_data.adata.obs["condition"] == "ctrl"]
)
print("Test Metrics:", test_metrics)

with open(f"{save_dir}/test_metrics.json", "w") as f:
    json.dump(test_metrics, f)
